# Predicting heart disease

Everything after the EDA notebooks: split the data, deal with the missing values,
check the distributions, encode, then train and tune tree models and an MLP.

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy import stats
from scipy.stats import (anderson, boxcox, kurtosis, levene, mannwhitneyu,
                         pointbiserialr, shapiro, skew)

from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler, RobustScaler
from sklearn.model_selection import (GridSearchCV, RandomizedSearchCV, StratifiedKFold,
                                     cross_val_score, train_test_split)
from sklearn.inspection import permutation_importance
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (GradientBoostingClassifier, RandomForestClassifier)
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_recall_curve,
                             precision_score, recall_score, roc_auc_score, roc_curve)

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

import helpers
from helpers import NUM_COLS, SEED

helpers.setup()

## Data and split

In [ ]:
data = helpers.load_data(zeros_as_missing=True)

X = data.drop(columns=['heartdisease'])
y = data['heartdisease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y, shuffle=True)

X_train.shape, X_test.shape

## 1. Is the missingness random?

If the rows with a missing cholesterol or blood pressure look like the complete rows,
a simple imputation would be fine. If they do not, the imputer has to use the other
features, which is what MICE does.

In [ ]:
train_has_missing = X_train.isna().any(axis=1)
print(f'Incomplete rows: {train_has_missing.sum()} of {len(X_train)}')

### Levene, on the spread

H0 is that the two groups have equal variances.

In [ ]:
rows = []
for col in NUM_COLS:
    missing = X_train.loc[train_has_missing, col].dropna()
    complete = X_train.loc[~train_has_missing, col].dropna()

    if len(missing) > 10 and len(complete) > 10:
        _, p_value = levene(missing, complete, center='median')
        rows.append({
            'feature': col,
            'var_missing': missing.var(),
            'var_complete': complete.var(),
            'p_value': p_value,
            'significant': p_value < 0.05,
        })

pd.DataFrame(rows).sort_values('p_value')

`restingbp`

- Variance (missing rows): ≈ **480**
- Variance (complete rows): ≈ **301**
- p-value: **0.004**

This is statistically strong evidence of unequal variance.

Interpretation:

- Rows with missing values show much wider spread in resting blood pressure
- Missingness is associated with heteroscedasticity
- Missingness is **not MCAR**

`age`

- Variance differs numerically, but
- p-value: 0.13

No strong statistical evidence of unequal variance.

That does not mean variances are equal — it means you can't confidently say they differ.

`maxhr`

- Variances are nearly identical
- p-value: 0.52

Missingness does not affect the spread of max heart rate.

`oldpeak`

- Variances are essentially identical
- p-value: 0.84

No association with missingness at all.

### Mann-Whitney, on the location

For each column that has gaps, compare the always-observed features between the rows
where it is missing and the rows where it is not. Mann-Whitney rather than a t-test
because none of these variables are normal.

In [ ]:
def missingness_report(df, missing_col):
    """Compare the fully observed numerical features against the gaps in missing_col."""
    is_missing = df[missing_col].isna()
    features = [col for col in df.select_dtypes('number').columns
                if col != missing_col and df[col].notna().all()]

    rows = []
    for col in features:
        when_missing = df.loc[is_missing, col]
        when_observed = df.loc[~is_missing, col]
        u_stat, p_value = mannwhitneyu(when_missing, when_observed, alternative='two-sided')

        rows.append({
            'feature': col,
            'mean_when_missing': when_missing.mean(),
            'mean_when_observed': when_observed.mean(),
            'mean_diff': when_missing.mean() - when_observed.mean(),
            'u_statistic': u_stat,
            'p_value': p_value,
            'significant': p_value < 0.05,
        })

    return pd.DataFrame(rows).sort_values('p_value')


for missing_col in X_train.columns[X_train.isna().any()]:
    gaps = X_train[missing_col].isna().sum()
    report = missingness_report(X_train, missing_col)
    related = report.loc[report['significant'], 'feature'].tolist()

    print(f'{missing_col}: {gaps} missing ({gaps / len(X_train):.1%})')
    print(report.to_string(index=False))
    print(f'Related to: {related if related else "nothing"}\n')

- `maxhr` (p < 0.05): There is a massive difference. The rows with missing data have a mean Max Heart Rate of 120, while complete rows have 140.

- `age` (p < 0.05): The rows with missing data are significantly older (56 years) than the complete rows (53 years).

It is missing specifically for older people with lower heart rates. This indicates a Missing At Random (MAR) mechanism (where the missingness depends on observed data like Age and Heart Rate).

Little's MCAR test would be the formal version of this — still to do.

## 2. Imputation

MAR rules out mean or median filling. MICE regresses each incomplete column on the
others and iterates, so it can use the age and heart-rate signal found above.
KNN was the other candidate but MICE kept the spread of cholesterol closer to the
observed values.

The imputer is fitted on the training set only and then applied to the test set.

In [ ]:
X_train_mice = X_train.copy()
X_test_mice = X_test.copy()

mice = IterativeImputer(max_iter=20, random_state=SEED,
                        sample_posterior=True, initial_strategy='constant')

X_train_mice[NUM_COLS] = mice.fit_transform(X_train[NUM_COLS])
X_test_mice[NUM_COLS] = mice.transform(X_test[NUM_COLS])

X_train_mice.isna().sum().sum(), X_test_mice.isna().sum().sum()

In [ ]:
fig, axes = plt.subplots(2, 2, sharex=True, figsize=(16, 8))

sns.histplot(X_train['cholesterol'], kde=True, stat='percent', ax=axes[0, 0])
axes[0, 0].set_title('Cholesterol before MICE')
sns.histplot(X_train_mice['cholesterol'], kde=True, stat='percent', ax=axes[1, 0])
axes[1, 0].set_title('Cholesterol after MICE')

sns.boxplot(X_train['cholesterol'], orient='h', width=.3, ax=axes[0, 1])
sns.boxplot(X_train_mice['cholesterol'], orient='h', width=.3, ax=axes[1, 1])

plt.tight_layout()
plt.show()

In [ ]:
# How far the imputation moved the centre and the spread.
before = X_train[helpers.ZERO_AS_MISSING]
after = X_train_mice[helpers.ZERO_AS_MISSING]

pd.DataFrame({
    'mean_before': before.mean(),
    'mean_after': after.mean(),
    'mean_delta': after.mean() - before.mean(),
    'std_before': before.std(),
    'std_after': after.std(),
    'std_delta': after.std() - before.std(),
})

## 3. Outliers

Two univariate rules on the imputed training set. IQR flags anything outside
`Q1 - 1.5·IQR` to `Q3 + 1.5·IQR`; the modified z-score uses the median and the median
absolute deviation instead of the mean and standard deviation, so a handful of extreme
values cannot inflate its own threshold.

The multivariate view (Mahalanobis distance) is in `3_eda_multivariate.ipynb`.

In [ ]:
def iqr_bounds(series, k=1.5):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    spread = q3 - q1
    return q1 - k * spread, q3 + k * spread


def modified_z_score(series):
    """Robust z-score based on the median absolute deviation."""
    median = series.median()
    mad = (series - median).abs().median()
    if np.isclose(mad, 0):
        return pd.Series(0.0, index=series.index)
    return 0.6745 * (series - median) / mad


def mad_bounds(series, threshold=3.5):
    median = series.median()
    mad = (series - median).abs().median()
    return median - mad / 0.6745 * threshold, median + mad / 0.6745 * threshold

In [ ]:
MAD_THRESHOLD = 3.5

rows = []
for col in NUM_COLS:
    series = X_train_mice[col]
    iqr_low, iqr_high = iqr_bounds(series)
    mad_low, mad_high = mad_bounds(series, MAD_THRESHOLD)

    iqr_flagged = (series < iqr_low) | (series > iqr_high)
    mad_flagged = modified_z_score(series).abs() > MAD_THRESHOLD

    rows.append({
        'feature': col,
        'iqr_lower': iqr_low,
        'iqr_upper': iqr_high,
        'iqr_outliers': iqr_flagged.sum(),
        'mad_lower': mad_low,
        'mad_upper': mad_high,
        'mad_outliers': mad_flagged.sum(),
        'percent': mad_flagged.mean() * 100,
    })

outliers = pd.DataFrame(rows)
outliers.round(2)

In [ ]:
# The rows the MAD rule flags, per feature.
for col in NUM_COLS:
    flagged = X_train_mice.loc[modified_z_score(X_train_mice[col]).abs() > MAD_THRESHOLD, col]
    if not flagged.empty:
        print(f'{col}: {sorted(flagged.round(1).tolist())}')

## 4. Shape of the distributions

In [ ]:
def distribution_summary(df, columns):
    rows = []
    for col in columns:
        series = df[col]
        rows.append({
            'feature': col,
            'mean': series.mean(),
            'median': series.median(),
            'std': series.std(),
            'min': series.min(),
            'max': series.max(),
            'skewness': skew(series, bias=False),
            'kurtosis': kurtosis(series, bias=False),
        })
    return pd.DataFrame(rows)


shape = distribution_summary(X_train_mice, NUM_COLS)
shape.round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, metric, thresholds in [(axes[0], 'skewness', (0.5, 1)),
                               (axes[1], 'kurtosis', (-1, 3))]:
    ax.barh(shape['feature'], shape[metric])
    ax.axvline(0, color='black', lw=1)
    for value in thresholds:
        ax.axvline(value, color='gray', ls='--', alpha=0.5)
        ax.axvline(-value, color='gray', ls='--', alpha=0.5)
    ax.set_xlabel(metric.capitalize())
    ax.set_title(f'{metric.capitalize()} by feature')

plt.tight_layout()
plt.show()

Anything with `|skewness| > 0.5` is worth a transformation, `> 1` definitely.

In [ ]:
skewed = shape.loc[shape['skewness'].abs() > 0.5, 'feature'].tolist()
skewed

### Do the usual transformations help?

In [ ]:
def candidate_transforms(series):
    """The transformations that are defined for this series."""
    transforms = {'original': series}
    if (series >= 0).all():
        transforms['log1p'] = np.log1p(series)
        transforms['sqrt'] = np.sqrt(series)
        transforms['reciprocal'] = 1 / (series + 1)
    if (series > 0).all():
        transforms['box-cox'] = pd.Series(boxcox(series)[0], index=series.index)
    return transforms


def compare_transforms(df, column):
    transforms = candidate_transforms(df[column])

    # squeeze=False keeps axes 2D even when only one transform applies (oldpeak).
    fig, axes = plt.subplots(1, len(transforms), figsize=(4 * len(transforms), 4),
                             squeeze=False)
    rows = []
    for ax, (name, values) in zip(axes[0], transforms.items()):
        skewness = skew(values, bias=False)
        rows.append({
            'transform': name,
            'skewness': skewness,
            'kurtosis': kurtosis(values, bias=False),
            'shapiro_p': shapiro(values).pvalue,
        })
        ax.hist(values, bins=30, alpha=0.7, edgecolor='black')
        ax.set_title(f'{name}\nskew: {skewness:.3f}')

    fig.suptitle(f'Transformations of {column}', fontsize=14)
    plt.tight_layout()
    plt.show()

    return pd.DataFrame(rows).sort_values('skewness', key=abs)


for col in skewed:
    print(f'--- {col} ---')
    print(compare_transforms(X_train_mice, col).to_string(index=False))

### Normality

Shapiro-Wilk and Anderson-Darling on the same columns, plus the Q-Q plots.
Tree models do not care about any of this, but it decides whether the parametric
tests further down are usable.

In [ ]:
def normality_table(df, columns):
    rows = []
    for col in columns:
        series = df[col].dropna()
        sw_stat, sw_p = shapiro(series)
        ad = anderson(series, dist='norm')
        # Index 2 of the critical values is the 5% level.
        ad_normal = ad.statistic < ad.critical_values[2]

        rows.append({
            'feature': col,
            'shapiro_stat': sw_stat,
            'shapiro_p': sw_p,
            'shapiro_normal': sw_p > 0.05,
            'anderson_stat': ad.statistic,
            'anderson_normal': ad_normal,
            'normal': (sw_p > 0.05) and ad_normal,
        })
    return pd.DataFrame(rows)


normality = normality_table(X_train_mice, NUM_COLS)
normality.round(4)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))

for ax, col in zip(axes.flatten(), NUM_COLS):
    stats.probplot(X_train_mice[col].dropna(), dist='norm', plot=ax)
    is_normal = normality.loc[normality['feature'] == col, 'normal'].iloc[0]
    p_value = normality.loc[normality['feature'] == col, 'shapiro_p'].iloc[0]
    ax.get_lines()[0].set_markersize(4)
    ax.set_title(f'{col}\n{"normal" if is_normal else "not normal"} (p={p_value:.4f})')

for ax in axes.flatten()[len(NUM_COLS):]:
    ax.set_visible(False)

fig.suptitle('Q-Q plots against the normal distribution', fontsize=14)
plt.tight_layout()
plt.show()

None of the features are normal, so the non-parametric tests above are the ones to trust.

## 5. Which features separate the two classes?

Point-biserial correlation for direction, Mann-Whitney for significance, and Cohen's d
for how big the gap actually is.

In [ ]:
def numeric_vs_target(X, y, columns):
    rows = []
    for col in columns:
        healthy = X.loc[y == 0, col]
        diseased = X.loc[y == 1, col]

        correlation, p_corr = pointbiserialr(y, X[col])
        _, p_mannwhitney = mannwhitneyu(healthy, diseased)
        pooled_std = np.sqrt((healthy.std() ** 2 + diseased.std() ** 2) / 2)

        rows.append({
            'feature': col,
            'correlation': correlation,
            'p_value_corr': p_corr,
            'mean_class_0': healthy.mean(),
            'mean_class_1': diseased.mean(),
            'mean_diff': diseased.mean() - healthy.mean(),
            'cohens_d': (diseased.mean() - healthy.mean()) / pooled_std,
            'p_value_mannwhitney': p_mannwhitney,
            'significant': p_mannwhitney < 0.05,
        })

    return pd.DataFrame(rows).sort_values('correlation', key=abs, ascending=False)


separation = numeric_vs_target(X_train_mice, y_train, NUM_COLS)
separation.round(4)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
colours = ['tab:red' if sig else 'lightgray' for sig in separation['significant']]

axes[0].barh(separation['feature'], separation['correlation'], color=colours)
axes[0].set_xlabel('Point-biserial correlation')
axes[0].set_title('Correlation with the target (red = p < 0.05)')

axes[1].barh(separation['feature'], separation['cohens_d'], color=colours)
for value, label in [(0.2, 'small'), (0.5, 'medium'), (0.8, 'large')]:
    axes[1].axvline(value, ls='--', alpha=0.5, label=label)
    axes[1].axvline(-value, ls='--', alpha=0.5)
axes[1].set_xlabel("Cohen's d")
axes[1].set_title('Effect size')
axes[1].legend()

for ax in axes:
    ax.axvline(0, color='black', lw=0.5)

plt.tight_layout()
plt.show()

## 6. Encoding

- `restingecg` and `chestpaintype` have no order, so one-hot.
- `st_slope` does have one (Down < Flat < Up), so ordinal.
- `exerciseangina` and `sex` are binary.
- `fastingbs` already comes in as 0/1.

The test set is reindexed onto the training columns afterwards, so a category that only
appears in one of the two splits cannot change the number of features.

In [ ]:
ST_SLOPE_ORDER = {'Down': 0, 'Flat': 1, 'Up': 2}
YES_NO = {'Y': 1, 'N': 0}


def encode(df, sex_encoder):
    encoded = pd.get_dummies(df, columns=['restingecg', 'chestpaintype'])
    encoded['st_slope_encoded'] = encoded['st_slope'].map(ST_SLOPE_ORDER)
    encoded['exerciseangina_encoded'] = encoded['exerciseangina'].map(YES_NO)
    encoded['sex_encoded'] = sex_encoder.transform(encoded['sex'])
    return encoded.drop(columns=['sex', 'st_slope', 'exerciseangina'])


sex_encoder = LabelEncoder().fit(X_train_mice['sex'])

X_train_encoded = encode(X_train_mice, sex_encoder)
X_test_encoded = encode(X_test_mice, sex_encoder).reindex(
    columns=X_train_encoded.columns, fill_value=0)

print(f'Train: {X_train_encoded.shape}, test: {X_test_encoded.shape}')
print(list(X_train_encoded.columns))

## 7. Feature engineering and scaling

The thresholds come from the clinical cut-offs rather than from the data:
cholesterol above 240 mg/dl, resting blood pressure above 140 mm Hg, max heart rate
below 120 bpm.

In [ ]:
def add_features(df):
    out = df.copy()

    out['age_group'] = pd.cut(out['age'], bins=[0, 30, 55, 70, 90],
                              labels=[1, 2, 3, 4]).astype(int)

    out['high_cholesterol'] = (out['cholesterol'] > 240).astype(int)
    out['high_bp'] = (out['restingbp'] > 140).astype(int)
    out['low_maxhr'] = (out['maxhr'] < 120).astype(int)

    out['risk_score'] = (out['high_cholesterol'] + out['high_bp']
                         + (out['age'] > 60).astype(int)
                         + (out['oldpeak'] > 1).astype(int))

    return out


X_train_engineered = add_features(X_train_encoded)
X_test_engineered = add_features(X_test_encoded)

print(f'{X_train_encoded.shape[1]} features -> {X_train_engineered.shape[1]}')

Only the continuous columns get scaled. The binary flags, the one-hot columns and the
small ordinal codes are left alone. The trees do not need any of this, the MLP does.

In [ ]:
SCALERS = {'standard': StandardScaler, 'minmax': MinMaxScaler, 'robust': RobustScaler}

NOT_SCALED = ['fastingbs', 'sex_encoded', 'exerciseangina_encoded', 'st_slope_encoded',
              'high_cholesterol', 'high_bp', 'low_maxhr', 'age_group', 'risk_score']


def scale(X_train, X_test, kind='standard'):
    leave_alone = set(NOT_SCALED) | {col for col in X_train.columns
                                     if col.startswith(('restingecg_', 'chestpaintype_'))}
    to_scale = [col for col in X_train.select_dtypes('number').columns
                if col not in leave_alone]

    scaler = SCALERS[kind]().fit(X_train[to_scale])

    X_train_scaled = X_train.copy()
    X_test_scaled = X_test.copy()
    X_train_scaled[to_scale] = scaler.transform(X_train[to_scale])
    X_test_scaled[to_scale] = scaler.transform(X_test[to_scale])

    return X_train_scaled, X_test_scaled, scaler, to_scale


X_train_scaled, X_test_scaled, scaler, scaled_cols = scale(
    X_train_engineered, X_test_engineered)

print(f'Scaled: {scaled_cols}')
X_train_scaled[scaled_cols].describe().T.round(3)

## 8. Tree models

In [ ]:
def evaluate(model, X_train, X_test, y_train, y_test, name):
    """Fit, predict on the test set, print the usual numbers and draw ROC + matrix."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    scores = {
        'model': name,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba),
    }

    print(name)
    for metric, value in scores.items():
        if metric != 'model':
            print(f'  {metric:<10} {value:.4f}')
    print(classification_report(y_test, y_pred, target_names=['No Disease', 'Disease']))

    plot_roc_and_matrix(y_test, y_pred, y_proba, name)
    return scores


def plot_roc_and_matrix(y_test, y_pred, y_proba, name):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    fpr, tpr, _ = roc_curve(y_test, y_proba)
    axes[0].plot(fpr, tpr, color='darkorange', lw=3,
                 label=f'AUC = {roc_auc_score(y_test, y_proba):.3f}')
    axes[0].plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
    axes[0].set_xlabel('False positive rate')
    axes[0].set_ylabel('True positive rate')
    axes[0].set_title(f'ROC - {name}')
    axes[0].legend(loc='lower right')

    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues',
                xticklabels=['No Disease', 'Disease'],
                yticklabels=['No Disease', 'Disease'], ax=axes[1])
    axes[1].set_xlabel('Predicted')
    axes[1].set_ylabel('Actual')
    axes[1].set_title(f'Confusion matrix - {name}')

    plt.tight_layout()
    plt.show()

### Cross-validated comparison

In [ ]:
SMALL = {
    'Decision Tree': DecisionTreeClassifier(max_depth=4, min_samples_leaf=1, random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=4, random_state=SEED),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=SEED),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=SEED, eval_metric='logloss'),
    'CatBoost': CatBoostClassifier(n_estimators=100, random_state=SEED, verbose=0),
}

BIG = {
    'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=2, random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=300, max_depth=5, min_samples_split=10,
                                            min_samples_leaf=1, random_state=SEED),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=300, random_state=SEED),
    'XGBoost': XGBClassifier(n_estimators=300, random_state=SEED, eval_metric='logloss'),
    'CatBoost': CatBoostClassifier(n_estimators=300, random_state=SEED, verbose=0),
}

folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

rows = []
for size, candidates in [('small', SMALL), ('big', BIG)]:
    for name, model in candidates.items():
        started = time.time()
        accuracy = cross_val_score(model, X_train_scaled, y_train, cv=folds, scoring='accuracy')
        roc_auc = cross_val_score(model, X_train_scaled, y_train, cv=folds, scoring='roc_auc')

        rows.append({
            'size': size,
            'model': name,
            'cv_accuracy': accuracy.mean(),
            'cv_accuracy_std': accuracy.std(),
            'cv_roc_auc': roc_auc.mean(),
            'seconds': time.time() - started,
        })

comparison = pd.DataFrame(rows)
comparison.sort_values('cv_roc_auc', ascending=False).round(4)

In [ ]:
# Does the bigger version of each model actually earn its extra trees?
comparison.pivot(index='model', columns='size', values='cv_accuracy').round(4)

### Baselines on the test set

In [ ]:
baselines = {
    'Decision Tree': DecisionTreeClassifier(max_depth=10, random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=SEED),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=10,
                                                    min_samples_split=10, min_samples_leaf=5,
                                                    random_state=SEED),
    'CatBoost': CatBoostClassifier(n_estimators=100, max_depth=6, learning_rate=0.01,
                                   subsample=0.8, random_state=SEED, verbose=0),
}

baseline_scores = [
    evaluate(model, X_train_scaled, X_test_scaled, y_train, y_test, name)
    for name, model in baselines.items()
]

pd.DataFrame(baseline_scores).round(4)

### What the forest is actually using

Impurity-based importance is biased towards high-cardinality features, so the
permutation version on the test set is the one to read.

In [ ]:
forest = RandomForestClassifier(n_estimators=100, random_state=SEED).fit(X_train_scaled, y_train)

impurity = pd.Series(forest.feature_importances_, index=X_train_scaled.columns)

permutation = permutation_importance(forest, X_test_scaled, y_test,
                                     n_repeats=20, random_state=SEED, n_jobs=-1)
permutation = pd.Series(permutation.importances_mean, index=X_test_scaled.columns)

importance = pd.DataFrame({'impurity': impurity, 'permutation': permutation})
importance = importance.sort_values('permutation', ascending=False)
importance.round(4)

In [ ]:
top = importance.head(15).iloc[::-1]

fig, axes = plt.subplots(1, 2, figsize=(14, 7), sharey=True)
axes[0].barh(top.index, top['impurity'], color='steelblue')
axes[0].set_title('Impurity importance')
axes[1].barh(top.index, top['permutation'], color='indianred')
axes[1].set_title('Permutation importance (test set)')

plt.tight_layout()
plt.show()

## 9. Tuning

In [ ]:
def tune(model, param_distributions, X_train, y_train, n_iter=20):
    search = RandomizedSearchCV(model, param_distributions, n_iter=n_iter,
                                cv=folds, scoring='roc_auc',
                                random_state=SEED, n_jobs=-1, verbose=1)
    search.fit(X_train, y_train)

    print(f'Best ROC-AUC: {search.best_score_:.4f}')
    print(f'Best parameters: {search.best_params_}')
    return search.best_estimator_

In [ ]:
best_rf = tune(
    RandomForestClassifier(random_state=SEED),
    {
        'n_estimators': [50, 100, 200, 300],
        'max_depth': [5, 10, 15, 20, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', None],
    },
    X_train_scaled, y_train)

In [ ]:
best_xgb = tune(
    XGBClassifier(random_state=SEED, eval_metric='logloss'),
    {
        'n_estimators': [50, 100, 200],
        'max_depth': [3, 5, 7, 9],
        'learning_rate': [0.01, 0.05, 0.1, 0.2],
        'subsample': [0.6, 0.8, 1.0],
        'colsample_bytree': [0.6, 0.8, 1.0],
        'gamma': [0, 0.1, 0.5],
    },
    X_train_scaled, y_train)

In [ ]:
tuned_scores = [
    evaluate(best_rf, X_train_scaled, X_test_scaled, y_train, y_test, 'Tuned Random Forest'),
    evaluate(best_xgb, X_train_scaled, X_test_scaled, y_train, y_test, 'Tuned XGBoost'),
]

pd.DataFrame(baseline_scores + tuned_scores).round(4)

## 10. Multilayer perceptron

Not a tree, but worth having as a comparison. Everything here runs on the scaled data.

In [ ]:
mlp_baseline = MLPClassifier(
    hidden_layer_sizes=(100, 50),
    activation='logistic',
    solver='adam',
    alpha=0.0001,
    learning_rate='adaptive',
    learning_rate_init=0.01,
    max_iter=500,
    validation_fraction=0.15,
    random_state=SEED,
)

evaluate(mlp_baseline, X_train_scaled, X_test_scaled, y_train, y_test, 'MLP (baseline)')

### One knob at a time

Everything is held at the baseline configuration except the setting under test, so the
differences are readable. A full grid comes next.

In [ ]:
MLP_DEFAULTS = {
    'hidden_layer_sizes': (100, 50),
    'activation': 'relu',
    'solver': 'adam',
    'learning_rate': 'adaptive',
    'learning_rate_init': 0.001,
    'alpha': 0.0001,
    'batch_size': 'auto',
    'max_iter': 500,
    'early_stopping': True,
    'n_iter_no_change': 10,
    'validation_fraction': 0.1,
    'random_state': SEED,
}

MLP_VARIANTS = {
    'learning rate': {
        'constant 0.001': {'learning_rate': 'constant', 'learning_rate_init': 0.001},
        'constant 0.01': {'learning_rate': 'constant', 'learning_rate_init': 0.01},
        'invscaling': {'learning_rate': 'invscaling', 'learning_rate_init': 0.001,
                       'power_t': 0.5},
        'adaptive 0.001': {'learning_rate': 'adaptive', 'learning_rate_init': 0.001},
        'adaptive 0.01': {'learning_rate': 'adaptive', 'learning_rate_init': 0.01},
    },
    'solver': {
        'adam': {'solver': 'adam'},
        'sgd': {'solver': 'sgd', 'learning_rate_init': 0.01, 'momentum': 0.9},
        'sgd + nesterov': {'solver': 'sgd', 'learning_rate_init': 0.01, 'momentum': 0.9,
                           'nesterovs_momentum': True},
        'lbfgs': {'solver': 'lbfgs'},
    },
    'activation': {
        name: {'activation': name} for name in ['relu', 'tanh', 'logistic', 'identity']
    },
    'regularisation': {
        f'alpha {value}': {'alpha': value} for value in [0.0, 0.0001, 0.001, 0.01, 0.1]
    },
    'architecture': {
        'shallow (100)': {'hidden_layer_sizes': (100,)},
        'medium (100, 50)': {'hidden_layer_sizes': (100, 50)},
        'deep (100, 50, 25)': {'hidden_layer_sizes': (100, 50, 25)},
        'wide (200, 100)': {'hidden_layer_sizes': (200, 100)},
        'pyramid (150, 100, 50, 25)': {'hidden_layer_sizes': (150, 100, 50, 25)},
        'narrow deep (50, 50, 50)': {'hidden_layer_sizes': (50, 50, 50)},
    },
    'batch size': {
        f'batch {value}': {'batch_size': value} for value in ['auto', 32, 64, 128]
    },
    'early stopping': {
        'patience 10': {'n_iter_no_change': 10},
        'patience 20': {'n_iter_no_change': 20},
        'patience 30': {'n_iter_no_change': 30},
        'off': {'early_stopping': False},
    },
}

In [ ]:
rows = []
for group, variants in MLP_VARIANTS.items():
    for name, overrides in variants.items():
        model = MLPClassifier(**{**MLP_DEFAULTS, **overrides})
        try:
            model.fit(X_train_scaled, y_train)
        except ValueError as error:
            print(f'{group} / {name} did not run: {error}')
            continue

        y_pred = model.predict(X_test_scaled)
        y_proba = model.predict_proba(X_test_scaled)[:, 1]
        train_accuracy = accuracy_score(y_train, model.predict(X_train_scaled))
        test_accuracy = accuracy_score(y_test, y_pred)

        rows.append({
            'group': group,
            'setting': name,
            'iterations': model.n_iter_,
            'train_accuracy': train_accuracy,
            'test_accuracy': test_accuracy,
            'precision': precision_score(y_test, y_pred),
            'recall': recall_score(y_test, y_pred),
            'f1': f1_score(y_test, y_pred),
            'roc_auc': roc_auc_score(y_test, y_proba),
            'overfitting': train_accuracy - test_accuracy,
        })

sweep = pd.DataFrame(rows)
sweep.sort_values('roc_auc', ascending=False).head(10).round(4)

In [ ]:
# Best setting within each group.
sweep.loc[sweep.groupby('group')['roc_auc'].idxmax()].round(4)

### Grid search

In [ ]:
mlp_grid = GridSearchCV(
    MLPClassifier(random_state=SEED),
    {
        'hidden_layer_sizes': [(50,), (100,), (100, 50), (100, 50, 25), (200, 100)],
        'activation': ['relu', 'tanh'],
        'solver': ['adam', 'sgd'],
        'alpha': [0.0001, 0.001, 0.01],
        'learning_rate': ['constant', 'adaptive'],
        'learning_rate_init': [0.001, 0.01],
        'max_iter': [500],
        'early_stopping': [True],
        'n_iter_no_change': [10, 20],
    },
    cv=folds, scoring='roc_auc', n_jobs=-1, verbose=2,
)

mlp_grid.fit(X_train_scaled, y_train)

print(f'Best CV ROC-AUC: {mlp_grid.best_score_:.4f}')
for param, value in mlp_grid.best_params_.items():
    print(f'  {param}: {value}')

In [ ]:
grid_results = pd.DataFrame(mlp_grid.cv_results_)
grid_results.nsmallest(10, 'rank_test_score')[
    ['rank_test_score', 'mean_test_score', 'std_test_score', 'params']
]

### Final model

The grid's best configuration, with a longer patience and a higher iteration cap so it
is allowed to converge properly.

In [ ]:
mlp = MLPClassifier(
    hidden_layer_sizes=(200, 100),
    activation='relu',
    solver='adam',
    learning_rate='adaptive',
    learning_rate_init=0.001,
    alpha=0.01,
    batch_size='auto',
    max_iter=2000,
    early_stopping=True,
    n_iter_no_change=50,
    validation_fraction=0.1,
    tol=1e-4,
    random_state=SEED,
)

mlp.fit(X_train_scaled, y_train)
print(f'Stopped after {mlp.n_iter_} iterations, final loss {mlp.loss_:.6f}')

In [ ]:
y_pred = mlp.predict(X_test_scaled)
y_proba = mlp.predict_proba(X_test_scaled)[:, 1]

train_accuracy = accuracy_score(y_train, mlp.predict(X_train_scaled))
test_accuracy = accuracy_score(y_test, y_pred)
test_precision = precision_score(y_test, y_pred)
test_recall = recall_score(y_test, y_pred)
test_f1 = f1_score(y_test, y_pred)

print(f'Train accuracy   {train_accuracy:.4f}')
print(f'Test accuracy    {test_accuracy:.4f}')
print(f'Test precision   {test_precision:.4f}')
print(f'Test recall      {test_recall:.4f}')
print(f'Test F1          {test_f1:.4f}')
print(f'Test ROC-AUC     {roc_auc_score(y_test, y_proba):.4f}')
print(f'Overfitting gap  {train_accuracy - test_accuracy:.4f}')

print(classification_report(y_test, y_pred,
                            target_names=['No Disease', 'Disease'], digits=4))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Final MLP', fontsize=16)

# ROC, with the threshold that maximises TPR - FPR.
fpr, tpr, thresholds = roc_curve(y_test, y_proba)
best = np.argmax(tpr - fpr)
axes[0, 0].plot(fpr, tpr, color='darkorange', lw=3,
                label=f'AUC = {roc_auc_score(y_test, y_proba):.3f}')
axes[0, 0].plot([0, 1], [0, 1], 'k--', lw=2, label='Random')
axes[0, 0].plot(fpr[best], tpr[best], 'ro', markersize=10,
                label=f'Best threshold = {thresholds[best]:.3f}')
axes[0, 0].set_xlabel('False positive rate')
axes[0, 0].set_ylabel('True positive rate')
axes[0, 0].set_title('ROC curve')
axes[0, 0].legend(loc='lower right')

matrix = confusion_matrix(y_test, y_pred)
labels = ['No Disease', 'Disease']

sns.heatmap(matrix, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=axes[0, 1])
axes[0, 1].set_xlabel('Predicted')
axes[0, 1].set_ylabel('Actual')
axes[0, 1].set_title('Confusion matrix')

sns.heatmap(matrix / matrix.sum(axis=1)[:, None], annot=True, fmt='.2%', cmap='Greens',
            xticklabels=labels, yticklabels=labels, ax=axes[0, 2])
axes[0, 2].set_xlabel('Predicted')
axes[0, 2].set_ylabel('Actual')
axes[0, 2].set_title('Confusion matrix, row-normalised')

precision, recall, _ = precision_recall_curve(y_test, y_proba)
axes[1, 0].plot(recall, precision, color='purple', lw=3, label=f'F1 = {test_f1:.3f}')
axes[1, 0].axhline(test_precision, color='r', ls='--',
                   label=f'Precision at 0.5 = {test_precision:.3f}')
axes[1, 0].axvline(test_recall, color='g', ls='--',
                   label=f'Recall at 0.5 = {test_recall:.3f}')
axes[1, 0].set_xlabel('Recall')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_title('Precision-recall curve')
axes[1, 0].legend(loc='lower left')

axes[1, 1].hist(y_proba[y_test == 0], bins=30, alpha=0.6, label='No disease',
                color='blue', edgecolor='black')
axes[1, 1].hist(y_proba[y_test == 1], bins=30, alpha=0.6, label='Disease',
                color='red', edgecolor='black')
axes[1, 1].axvline(0.5, color='black', ls='--', lw=2, label='Threshold')
axes[1, 1].set_xlabel('Predicted probability')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Predicted probability by true class')
axes[1, 1].legend()

axes[1, 2].plot(mlp.loss_curve_, lw=2, color='darkblue')
axes[1, 2].set_xlabel('Iteration')
axes[1, 2].set_ylabel('Loss')
axes[1, 2].set_title(f'Training loss (final {mlp.loss_:.6f})')

plt.tight_layout()
plt.show()

In [ ]:
tn, fp, fn, tp = matrix.ravel()

print(f'True negatives  {tn:4d}  correctly predicted no disease')
print(f'False positives {fp:4d}  predicted disease, patient was healthy')
print(f'False negatives {fn:4d}  predicted healthy, patient had disease')
print(f'True positives  {tp:4d}  correctly predicted disease')
print(f'\nCorrect: {tn + tp} of {matrix.sum()} ({(tn + tp) / matrix.sum():.2%})')